# RL: Direct Preference Optimization (DPO)

## Task Description
Fine-tune an LLM to align with human preferences using DPO (Direct Preference Optimization), a stable alternative to RLHF.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## Learning Objectives

*   **Practical:** Gain hands-on experience with this model.
*   **Theory:** Understand the underlying concepts and architecture.

In [ ]:
# Check for GPU
!nvidia-smi

## 1. Installation
Install necessary libraries.

In [ ]:
!pip install trl peft bitsandbytes accelerate datasets transformers -q

## 2. Imports
Import libraries.

In [ ]:
from transformers import AutoTokenizer, TrainingArguments
from trl import DPOTrainer
from datasets import load_dataset

## 3. Load Data
Load dataset or input data.

In [ ]:
# Load a small DPO dataset
dataset = load_dataset("trilogy/queer-diff", split="train[:100]") # Small subset for demo
print(f"Loaded {len(dataset)} examples.")
print(dataset[0])

## 4. Data Cleaning
Preprocess the data.

In [ ]:
# Format dataset for DPO (prompt, chosen, rejected)
def process(row):
    return {
        "prompt": row["prompt"],
        "chosen": row["chosen"],
        "rejected": row["rejected"]
    }
dataset = dataset.map(process)

## 5. Load Model
Load the model from Hugging Face.

In [ ]:
# Use a tiny model for demonstration purposes on Colab
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Loaded {model_id}")

## 6. Inference
Run the model.

In [ ]:
# We simulate a DPO training setup (Inference here is effectively 'Run Training')
# Note: Real training takes time. We set steps to 1 for demonstration.
training_args = TrainingArguments(
    output_dir="./dpo_results",
    per_device_train_batch_size=1,
    max_steps=1, # Demo only
    learning_rate=1e-3,
    logging_steps=1,
    report_to="none"
)

trainer = DPOTrainer(
    model=model_id, # passing ID allows TRL to load it
    ref_model=None, # TRL will load a copy
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    beta=0.1,
)

print("Starting DPO step...")
trainer.train()
results = "Training complete (1 step demo)."
print(results)

## 7. Save Results
Save outputs.

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage
Explore further.

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print('Try different inputs!')

## 9. Troubleshooting

*   **Out of Memory (OOM):** If you encounter OOM errors, try reducing the batch size or input length. On the Free Tier, you are limited to T4 GPUs with 16GB VRAM.
*   **GPU Not Found:** Ensure your runtime type is set to GPU. Go to *Runtime > Change runtime type* and select *T4 GPU*.
*   **Missing Packages:** If you see `ModuleNotFoundError`, make sure you have run the **Installation** step (Step 1).